# Experiment: Hyperparameter Tuning and Model Selection

In a professional machine learning pipeline, finding the optimal configuration for a model is a critical step. Since the performance of a Fuzzy Rule-Based Classifier depends heavily on its structure, we must systematically evaluate different settings to ensure the model generalizes well to new data.
The Methodology: Train/Validation/Test Split

To avoid biased results and ensure a robust evaluation, we divide our dataset into three independent subsets:

-  Training Set: Used by the evolutionary algorithm to discover and refine the fuzzy rules.

-  Validation Set: Used as a "tuning ground." We evaluate different hyperparameter combinations here to select the best-performing version of the model.

-  Test Set: Held out until the very end. This set provides the final, unbiased accuracy of the selected model on completely unseen data.

Search Space

In this experiment, we focus on tuning two fundamental structural parameters:

-    nRules: The total number of fuzzy rules the system is allowed to evolve.

-    nAnts: The maximum number of antecedents (conditions) per rule, which controls the complexity of the logic.

-    Note: While we are focusing on these two variables, this same methodology can be applied to any other parameter in the Ex-Fuzzy library, such as the number of generations (n_gen), the population size (pop_size), or the configuration of the linguistic variables.

Objective

The goal is to find the "sweet spot" where the model is complex enough to capture the patterns in the data but simple enough to remain interpretable and avoid overfitting. We will select the combination that yields the highest Validation Accuracy and then verify its true performance using the Test Set.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_iris

from ex_fuzzy.fuzzy_sets import FUZZY_SETS
import ex_fuzzy.evolutionary_fit as GA
from ex_fuzzy.utils import construct_partitions

In [ ]:

iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target


fz_type_studied = FUZZY_SETS.t1
vl = 3
linguistic_variables = construct_partitions(X, fz_type_studied)

# 1. Data Splitting (70% Train, 15% Val, 15% Test)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.176, random_state=42)

# 2. Define the Search Space (Hyperparameters)
rules_options = [5, 10, 15]
ants_options = [2, 4]

best_val_acc = 0
best_params = {}
best_model = None

print("Starting parameter tuning...")
print("-" * 40)

# 3. Tuning Loop (Manual Grid Search)
for n_rules in rules_options:
    for n_ants in ants_options:
        # Initialize the model with the current combination

        model = GA.BaseFuzzyRulesClassifier(
            nRules=n_rules,
            nAnts=n_ants,
            linguistic_variables=linguistic_variables,
            n_linguistic_variables=vl,
            fuzzy_type=fz_type_studied, 
            verbose=False
            )
        
        # Train on the TRAINING set
        model.fit(X_train, y_train, n_gen=20, pop_size=30)
        
        # Evaluate on the VALIDATION set
        y_val_pred = model.predict(X_val)
        val_acc = accuracy_score(y_val, y_val_pred)
        
        print(f"Testing: nRules={n_rules}, nAnts={n_ants} -> Val Acc: {val_acc:.4f}")
        
        # Save the best model based on VALIDATION performance
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_params = {'nRules': n_rules, 'nAnts': n_ants}
            best_model = model

# 4. Final Evaluation on the TEST set
print("-" * 40)
print(f"Best parameters found: {best_params}")
print(f"Validation Accuracy: {best_val_acc:.4f}")

# The final "moment of truth" using the unseen Test set
y_test_pred = best_model.predict(X_test)
final_acc = accuracy_score(y_test, y_test_pred)

print(f"FINAL PERFORMANCE (Test Accuracy): {final_acc:.4f}")